# Data Exploration

In [2]:
!pip install numpy
!pip install pandas
!pip install scanpy
!pip install anndata


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  Using cached scanpy-1.12.4-py3-none-any.whl.metadata (8.5 kB)
  Using cached fast_array_utils-1.5.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached patsy-1.0.3-py2.py3-none-any.whl.metadata (3.7 kB)
  Using cached pynndescent-0.6.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached statsmodels-0.15.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.6 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
Using cached scanpy-1.12.4-py3-none-any.whl (2.2 MB)
Using cached fast_array_utils-1.5.1-py3-none-any.whl (40 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1

In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

In [3]:
#navigate to the Reveal_Genomics_Fall_2025 folder path
#%cd /content/drive/MyDrive/Reveal_Genomics_Fall_2025


In [4]:
tcga_df = pd.read_csv("TCGA_PanCancer_data.tsv", sep="\t")
metadata_df = pd.read_csv("merged_sample_quality_annotations.tsv", sep="\t")

FileNotFoundError: [Errno 2] No such file or directory: 'TCGA_PanCancer_data.tsv'

In [ ]:
tcga_df

In [ ]:
tcga_df = tcga_df.set_index("gene_id")
tcga_df = tcga_df.T

In [ ]:
nan_columns = tcga_df.columns[tcga_df.isna().any()].tolist()
nan_columns

In [ ]:
print(tcga_df.isna().sum())

In [ ]:
#Dropping all columns with na values
tcga_df = tcga_df.dropna(axis=1)

# Check for any remaining infinite or NaN values and handle them
tcga_df = tcga_df.replace([np.inf, -np.inf], np.nan)
tcga_df = tcga_df.fillna(0)  # or use tcga_df = tcga_df.dropna()

In [ ]:
tcga_df

In [ ]:
metadata_df

In [ ]:
included_cells = []
for cell in metadata_df["aliquot_barcode"]:
  if cell in tcga_df.index:
    included_cells.append(cell)

In [ ]:
included_cells = np.unique(included_cells)
filtered_metadata = metadata_df[metadata_df["aliquot_barcode"].isin(included_cells)]
filtered_metadata = filtered_metadata.drop_duplicates(subset="aliquot_barcode", keep="first")

In [ ]:
filtered_metadata

In [ ]:
filtered_metadata = filtered_metadata.set_index("aliquot_barcode")

In [ ]:
filtered_metadata

In [ ]:
filtered_metadata = filtered_metadata.loc[tcga_df.index]
filtered_metadata

In [ ]:
adata_tcga = ad.AnnData(tcga_df)
adata_tcga.obs = filtered_metadata

In [ ]:
adata_tcga.layers["raw_counts"] = adata_tcga.X.copy()

In [ ]:
sc.pp.normalize_total(adata_tcga)
sc.pp.log1p(adata_tcga)

In [ ]:
adata_tcga

#Autoencoder

Define the Autoencoder

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
import anndata as ad

# ----- Autoencoder model -----
class AE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden=[1024, 256, 64], dropout=0.1):
        super().__init__()
        enc_layers = []
        last = input_dim
        for h in hidden:
            enc_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        enc_layers += [nn.Linear(last, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        last = latent_dim
        for h in reversed(hidden):
            dec_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        dec_layers += [nn.Linear(last, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

# ----- Load data -----
# adata = ad.read_h5ad("adata_tcga_preprocessed.h5ad") # This line is not needed as adata_tcga is already loaded
X = adata_tcga.X # Use the existing adata_tcga object
if not isinstance(X, np.ndarray):
    X = X.A if hasattr(X, "A") else X.toarray()
X = X.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

In [ ]:
adata_tcga.write_h5ad("adata_tcga_preprocessed.h5ad", compression="lzf")

#**IMPORTANT**: Preparing and Uploading the `.h5ad` File for Colab

## Step 1: Save the AnnData object in your preprocessing notebook

At the end of your `DataExploration.ipynb`, add the following code to save your cleaned and log-normalized dataset:

```python
import anndata as ad

# Save preprocessed AnnData object
adata_tcga.write_h5ad("adata_tcga_preprocessed.h5ad", compression="lzf")
```

This will create a file called **`adata_tcga_preprocessed.h5ad`** in your local directory.
It contains:

* `adata_tcga.X`: log-normalized gene expression matrix
* `adata_tcga.obs`: sample metadata (cancer type, annotations, etc.)
* `adata_tcga.var`: gene metadata

---

## Step 2: Upload the `.h5ad` file to Colab

In your Colab notebook, run the following code cell and select the saved `.h5ad` file from your computer:

```python
from google.colab import files

# Upload h5ad file into Colab
uploaded = files.upload()

# Check that the file is present
!ls -lh
```

---

## Step 3: Load the `.h5ad` file in Colab

Once uploaded, you can load it with AnnData:

```python
import anndata as ad

# Load the dataset
adata = ad.read_h5ad("adata_tcga_preprocessed.h5ad")

# Inspect the structure
print(adata)
print(adata.X.shape)   # (samples × genes)
print(adata.obs.head())  # metadata
```

---

You now have the **preprocessed TCGA dataset in Colab** and can use it for training your Autoencoder or Variational Autoencoder.

In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
from sklearn.preprocessing import StandardScaler

# Load the dataset
adata = ad.read_h5ad("adata_tcga_preprocessed.h5ad")

print(f"Initial shape: {adata.shape}")
print(f"Total NaN values in X: {np.isnan(adata.X).sum()}")

# Remove genes with ANY NaN from X
# Convert to dense array to check for NaN
if hasattr(adata.X, 'toarray'):
    X_dense = adata.X.toarray()
else:
    X_dense = adata.X.copy()

genes_with_nan = np.isnan(X_dense).any(axis=0)
print(f"Genes with NaN: {np.sum(genes_with_nan)}")

# Keep only genes without NaN
adata = adata[:, ~genes_with_nan]
print(f"After removing genes with NaN: {adata.shape}")

# Remove samples with NaN cancer type (optional, but good practice)
adata = adata[~adata.obs['cancer type'].isna(), :]
print(f"Final shape after all cleaning: {adata.shape}")

# Verify no NaN remain in X
if hasattr(adata.X, 'toarray'):
    X_dense = adata.X.toarray()
else:
    X_dense = adata.X.copy()
print(f"NaN remaining in X: {np.isnan(X_dense).sum()}")

In [ ]:
# Extract X for training
X = adata.X
if not isinstance(X, np.ndarray):
    X = X.A if hasattr(X, "A") else X.toarray()
X = X.astype(np.float32)

# Final check before scaling
print(f"Before scaling - NaN: {np.isnan(X).sum()}, Inf: {np.isinf(X).sum()}")

# Scale the data
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

print(f"After scaling - NaN: {np.isnan(X).sum()}, Inf: {np.isinf(X).sum()}")

Train Autoencoder

In [ ]:
def train_autoencoder(X, latent_dim=32, hidden=[1024,256,64],
                      dropout=0.1, batch_size=256, lr=1e-3,
                      weight_decay=1e-5, epochs=50, val_split=0.1, seed=42):

    torch.manual_seed(seed)
    np.random.seed(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_cells, n_genes = X.shape

    model = AE(input_dim=n_genes, latent_dim=latent_dim, hidden=hidden, dropout=dropout).to(device)

    X_tensor = torch.from_numpy(X)
    dataset = TensorDataset(X_tensor)
    n_val = int(len(dataset) * val_split)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(seed))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs+1):
        # Training
        model.train()
        train_loss = 0.0
        for (xb,) in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat, _ = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= n_train

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(device)
                x_hat, _ = model(xb)
                loss = criterion(x_hat, xb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= n_val

        print(f"Epoch {epoch:03d}/{epochs}  train MSE: {train_loss:.6f}  val MSE: {val_loss:.6f}")

    # Extract embeddings
    model.eval()
    with torch.no_grad():
        Z = []
        for i in range(0, X.shape[0], 4096):
            xb = torch.from_numpy(X[i:i+4096]).to(device)
            _, z = model(xb)
            Z.append(z.cpu().numpy())
        Z = np.vstack(Z).astype(np.float32)

    return model, Z

# Train
model, Z = train_autoencoder(X, latent_dim=32, epochs=50)


Save embeddings back to AnnData

In [ ]:
adata.obsm["X_ae_32"] = Z
adata.write_h5ad("adata_with_X_ae_32.h5ad", compression="lzf")

pd.DataFrame(Z, index=adata.obs_names).to_csv("embeddings_latent32.csv")

Visualize with UMAP

In [ ]:
import scanpy as sc

sc.pp.neighbors(adata, use_rep="X_ae_32")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["cancer type"], wspace=0.4)

Download results

In [ ]:
#from google.colab import files
#files.download("embeddings_latent32.csv")
#files.download("adata_withh5ad_X_ae_32.")

#VAE Model

Define the VAE Model

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
import anndata as ad

# ---- VAE ----
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden=[1024, 256, 64], dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        enc_layers = []
        last = input_dim
        for h in hidden:
            enc_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        self.encoder = nn.Sequential(*enc_layers)

        self.fc_mu = nn.Linear(last, latent_dim)
        self.fc_logvar = nn.Linear(last, latent_dim)

        # Decoder
        dec_layers = []
        last = latent_dim
        for h in reversed(hidden):
            dec_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        dec_layers += [nn.Linear(last, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar, z

def vae_loss(x_hat, x, mu, logvar):
    recon = nn.functional.mse_loss(x_hat, x, reduction="sum")
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + kld

Train the VAE Mode

In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
from sklearn.preprocessing import StandardScaler

# Load the dataset
adata = ad.read_h5ad("adata_tcga_preprocessed.h5ad")

print(f"Initial shape: {adata.shape}")
print(f"Total NaN values in X: {np.isnan(adata.X).sum()}")

# Remove genes with ANY NaN from X
# Convert to dense array to check for NaN
if hasattr(adata.X, 'toarray'):
    X_dense = adata.X.toarray()
else:
    X_dense = adata.X.copy()

genes_with_nan = np.isnan(X_dense).any(axis=0)
print(f"Genes with NaN: {np.sum(genes_with_nan)}")

# Keep only genes without NaN
adata = adata[:, ~genes_with_nan]
print(f"After removing genes with NaN: {adata.shape}")

# Remove samples with NaN cancer type (optional, but good practice)
adata = adata[~adata.obs['cancer type'].isna(), :]
print(f"Final shape after all cleaning: {adata.shape}")

# Verify no NaN remain in X
if hasattr(adata.X, 'toarray'):
    X_dense = adata.X.toarray()
else:
    X_dense = adata.X.copy()
print(f"NaN remaining in X: {np.isnan(X_dense).sum()}")

# Extract X for training
X = adata.X
if not isinstance(X, np.ndarray):
    X = X.A if hasattr(X, "A") else X.toarray()
X = X.astype(np.float32)

# Final check before scaling
print(f"Before scaling - NaN: {np.isnan(X).sum()}, Inf: {np.isinf(X).sum()}")

# Scale the data
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

print(f"After scaling - NaN: {np.isnan(X).sum()}, Inf: {np.isinf(X).sum()}")

In [ ]:
# Training
def train_vae(X, latent_dim=32, hidden=[1024,256,64], batch_size=256, lr=1e-3,
              epochs=50, val_split=0.1, seed=42):

    torch.manual_seed(seed)
    np.random.seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    n_cells, n_genes = X.shape
    model = VAE(input_dim=n_genes, latent_dim=latent_dim, hidden=hidden).to(device)

    X_tensor = torch.from_numpy(X)
    dataset = TensorDataset(X_tensor)
    n_val = int(len(dataset) * val_split)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0
        for (xb,) in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat, mu, logvar, _ = model(xb)
            loss = vae_loss(x_hat, xb, mu, logvar)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        print(f"Epoch {epoch:03d}/{epochs}  Train Loss: {train_loss/len(train_ds):.6f}")

    # Latent embeddings
    model.eval()
    Z = []
    with torch.no_grad():
        for i in range(0, X.shape[0], 4096):
            xb = torch.from_numpy(X[i:i+4096]).to(device)
            _, mu, _, _ = model(xb)
            Z.append(mu.cpu().numpy())
    Z = np.vstack(Z).astype(np.float32)
    return model, Z

# Train
vae_model, Z = train_vae(X, latent_dim=32, epochs=50)

Save embeddings back to AnnData

In [ ]:
adata.obsm["X_vae_32"] = Z
adata.write_h5ad("adata_with_X_vae_32.h5ad", compression="lzf")

pd.DataFrame(Z, index=adata.obs_names).to_csv("vae_embeddings_latent32.csv")

Visualize

In [ ]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

# UMAP (Figure 2A/B)
sc.pp.neighbors(adata, use_rep="X_vae_32")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["cancer type"], title="VAE Latent UMAP by Cancer Type")

# Heatmap (Figure 2C)
latent_df = pd.DataFrame(adata.obsm["X_vae_32"], index=adata.obs_names)
sns.clustermap(latent_df, row_cluster=True, col_cluster=True, cmap="RdYlBu_r", figsize=(12,10))
plt.show()


# Description of the Work

I implemented a **Variational Autoencoder (VAE)** to learn a low-dimensional latent representation of the TCGA Pan-Cancer RNA-seq dataset.

---

## Data Preprocessing

* Gene expression data (16,335 features per tumor sample) was preprocessed by:

  * Removing missing values
  * Filtering metadata
  * Log-normalizing counts
  * Merging expression data with sample metadata
* The final dataset was stored in an **AnnData** object (`adata_tcga`) and exported as a `.h5ad` file for downstream analysis.

---

## Model Architecture

* The VAE consists of:

  * **Encoder**: compresses high-dimensional gene expression profiles into a **32-dimensional latent space**.
  * **Latent Layer**: outputs both a mean (`μ`) and log-variance (`logσ²`) for each latent variable. Using the **reparameterization trick**, latent embeddings are sampled.
  * **Decoder**: reconstructs the original expression profiles from the latent space.

---

## Loss Function

Training minimized a combined loss of:

1. **Reconstruction Loss**: Mean squared error between the input and reconstructed output.
2. **Kullback–Leibler Divergence (KL)**: Forces the latent distribution to align with a unit Gaussian prior, improving generalizability and structure of the latent space.

---

## Training

* Optimizer: **Adam**
* Regularization: **Dropout**
* Mini-batch training with a **validation split**
* Model trained for 50 epochs

---

## Results (Replicating Figure 2 from Tybalt VAE Paper)

* **UMAP Projections (Figure 2A/B):**
  Latent embeddings projected into 2D using UMAP showed that samples clustered by cancer type. This indicates that the VAE captured biologically meaningful patterns in the data.

* **Heatmap of Encodings (Figure 2C):**
  A heatmap of the latent encodings revealed structured variation across tumors. Hierarchical clustering highlighted groups of samples with similar expression programs, consistent with distinct cancer-type biology.

---

## Significance

The VAE successfully extracted a **compact latent space** where cancer types formed coherent clusters. This demonstrates the ability of deep generative models to uncover biologically relevant features from large-scale gene expression datasets.

This work mirrors the results of the **Tybalt VAE** paper (PubMed ID: [29218871](https://pubmed.ncbi.nlm.nih.gov/29218871/)), confirming that VAEs can be powerful tools for cancer stratification and for modeling gene expression landscapes.

# Task
Train the autoencoder using the `X` data, and then create a pandas DataFrame containing the generated 32-dimensional latent embeddings (`Z`), using `adata_tcga.obs_names` as the index.

## Train Autoencoder and Generate Embeddings

### Subtask:
Execute the `train_autoencoder` function with the preprocessed data (X) to obtain the 32-dimensional latent embeddings (Z) for all cells.


**Reasoning**:
The user wants to execute the `train_autoencoder` function with the preprocessed data `X`, specifying `latent_dim=32` and `epochs=50`, and store the results in `model` and `Z`. This directly addresses the subtask instructions.



In [ ]:
def train_autoencoder(X, latent_dim=32, hidden=[1024,256,64],
                      dropout=0.1, batch_size=256, lr=1e-3,
                      weight_decay=1e-5, epochs=50, val_split=0.1, seed=42):

    torch.manual_seed(seed)
    np.random.seed(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_cells, n_genes = X.shape

    model = AE(input_dim=n_genes, latent_dim=latent_dim, hidden=hidden, dropout=dropout).to(device)

    X_tensor = torch.from_numpy(X)
    dataset = TensorDataset(X_tensor)
    n_val = int(len(dataset) * val_split)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(seed))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs+1):
        # Training
        model.train()
        train_loss = 0.0
        for (xb,) in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat, _ = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= n_train

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(device)
                x_hat, _ = model(xb)
                loss = criterion(x_hat, xb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= n_val

        print(f"Epoch {epoch:03d}/{epochs}  train MSE: {train_loss:.6f}  val MSE: {val_loss:.6f}")

    # Extract embeddings
    model.eval()
    with torch.no_grad():
        Z = []
        for i in range(0, X.shape[0], 4096):
            xb = torch.from_numpy(X[i:i+4096]).to(device)
            _, z = model(xb)
            Z.append(z.cpu().numpy())
        Z = np.vstack(Z).astype(np.float32)

    return model, Z

# Train
model, Z = train_autoencoder(X, latent_dim=32, epochs=50)

**Reasoning**:
The previous code failed because the `X` variable, which holds the preprocessed data, was not defined before `train_autoencoder` was called. I also need to ensure the `AE` class is defined. I will include the necessary code to define the `AE` class, load and preprocess `X`, define the `train_autoencoder` function, and then call it, all within a single code block to ensure proper execution order and variable availability.



In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
import anndata as ad

# ----- Autoencoder model -----
class AE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden=[1024, 256, 64], dropout=0.1):
        super().__init__()
        enc_layers = []
        last = input_dim
        for h in hidden:
            enc_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        enc_layers += [nn.Linear(last, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        last = latent_dim
        for h in reversed(hidden):
            dec_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        dec_layers += [nn.Linear(last, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

def train_autoencoder(X, latent_dim=32, hidden=[1024,256,64],
                      dropout=0.1, batch_size=256, lr=1e-3,
                      weight_decay=1e-5, epochs=50, val_split=0.1, seed=42):

    torch.manual_seed(seed)
    np.random.seed(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_cells, n_genes = X.shape

    model = AE(input_dim=n_genes, latent_dim=latent_dim, hidden=hidden, dropout=dropout).to(device)

    X_tensor = torch.from_numpy(X)
    dataset = TensorDataset(X_tensor)
    n_val = int(len(dataset) * val_split)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(seed))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs+1):
        # Training
        model.train()
        train_loss = 0.0
        for (xb,) in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat, _ = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= n_train

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(device)
                x_hat, _ = model(xb)
                loss = criterion(x_hat, xb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= n_val

        print(f"Epoch {epoch:03d}/{epochs}  train MSE: {train_loss:.6f}  val MSE: {val_loss:.6f}")

    # Extract embeddings
    model.eval()
    with torch.no_grad():
        Z = []
        for i in range(0, X.shape[0], 4096):
            xb = torch.from_numpy(X[i:i+4096]).to(device)
            _, z = model(xb)
            Z.append(z.cpu().numpy())
        Z = np.vstack(Z).astype(np.float32)

    return model, Z

# Prepare data (X) for training
# This line is not needed as adata_tcga is already loaded from previous cells
# adata = ad.read_h5ad("adata_tcga_preprocessed.h5ad")
X = adata_tcga.X # Use the existing adata_tcga object
if not isinstance(X, np.ndarray):
    X = X.A if hasattr(X, "A") else X.toarray()
X = X.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

# Train
model, Z = train_autoencoder(X, latent_dim=32, epochs=50)

**Reasoning**:
The previous error occurred because the `AE` class and `X` variable were not defined within the scope where `train_autoencoder` was called. The provided code block correctly defines `AE` and `train_autoencoder`, prepares the `X` data from `adata_tcga`, and then calls `train_autoencoder` to generate the latent embeddings `Z`. This combined code block ensures all necessary components are available and executed in the correct order to resolve the `NameError`.



In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
import anndata as ad

# ----- Autoencoder model -----
class AE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden=[1024, 256, 64], dropout=0.1):
        super().__init__()
        enc_layers = []
        last = input_dim
        for h in hidden:
            enc_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        enc_layers += [nn.Linear(last, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        last = latent_dim
        for h in reversed(hidden):
            dec_layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        dec_layers += [nn.Linear(last, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

def train_autoencoder(X, latent_dim=32, hidden=[1024,256,64],
                      dropout=0.1, batch_size=256, lr=1e-3,
                      weight_decay=1e-5, epochs=50, val_split=0.1, seed=42):

    torch.manual_seed(seed)
    np.random.seed(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_cells, n_genes = X.shape

    model = AE(input_dim=n_genes, latent_dim=latent_dim, hidden=hidden, dropout=dropout).to(device)

    X_tensor = torch.from_numpy(X)
    dataset = TensorDataset(X_tensor)
    n_val = int(len(dataset) * val_split)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(seed))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(1, epochs+1):
        # Training
        model.train()
        train_loss = 0.0
        for (xb,) in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat, _ = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= n_train

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (xb,) in val_loader:
                xb = xb.to(device)
                x_hat, _ = model(xb)
                loss = criterion(x_hat, xb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= n_val

        print(f"Epoch {epoch:03d}/{epochs}  train MSE: {train_loss:.6f}  val MSE: {val_loss:.6f}")

    # Extract embeddings
    model.eval()
    with torch.no_grad():
        Z = []
        for i in range(0, X.shape[0], 4096):
            xb = torch.from_numpy(X[i:i+4096]).to(device)
            _, z = model(xb)
            Z.append(z.cpu().numpy())
        Z = np.vstack(Z).astype(np.float32)

    return model, Z

# Prepare data (X) for training
# adata = ad.read_h5ad("adata_tcga_preprocessed.h5ad") # This line is not needed as adata_tcga is already loaded from previous cells
X = adata_tcga.X # Use the existing adata_tcga object
if not isinstance(X, np.ndarray):
    X = X.A if hasattr(X, "A") else X.toarray()
X = X.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

# Train
model, Z = train_autoencoder(X, latent_dim=32, epochs=50)

In [4]:
import anndata as ad
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

adata_tcga = ad.read_h5ad("adata_tcga_preprocessed.h5ad")
clinical = pd.read_csv(
    "Survival_SupplementalTable_S1_20171025_xena_sp",
    sep="\t",
)

In [5]:
sample_patient_ids = (
    adata_tcga.obs["patient_barcode"].astype(str).reset_index(drop=True)
)
barcode_patient_ids = (
    pd.Series(adata_tcga.obs_names.astype(str))
    .str[:12]
    .reset_index(drop=True)
)

assert sample_patient_ids.equals(barcode_patient_ids)
print("Expression patients:", sample_patient_ids.nunique())
print("Patients with multiple samples:", int((sample_patient_ids.value_counts() > 1).sum()))

clinical_check = clinical.copy()
clinical_check["_PATIENT"] = clinical_check["_PATIENT"].astype(str)
clinical_check["OS"] = pd.to_numeric(clinical_check["OS"], errors="coerce")
clinical_check["OS.time"] = pd.to_numeric(clinical_check["OS.time"], errors="coerce")

clinical_consistency = clinical_check.groupby("_PATIENT").agg(
    n_os=("OS", "nunique"),
    n_os_time=("OS.time", "nunique"),
)
print("Patients with conflicting OS:", int((clinical_consistency["n_os"] > 1).sum()))
print("Patients with conflicting OS.time:", int((clinical_consistency["n_os_time"] > 1).sum()))

assert not (clinical_consistency["n_os"] > 1).any()
assert not (clinical_consistency["n_os_time"] > 1).any()

Expression patients: 10274
Patients with multiple samples: 785
Patients with conflicting OS: 0
Patients with conflicting OS.time: 0


In [6]:
# GroupBy.first skips missing values, so a missing first duplicate
# won't discard a later non-missing outcome for that patient.
clinical_patient = (
    clinical_check.groupby("_PATIENT", as_index=False)
    .agg(OS=("OS", "first"), OS_time=("OS.time", "first"))
)

clinical_patient = clinical_patient.dropna(subset=["OS", "OS_time"])
clinical_patient = clinical_patient[
    clinical_patient["OS"].isin([0, 1]) & (clinical_patient["OS_time"] > 0)
]

expression_ids = set(adata_tcga.obs["patient_barcode"].astype(str))
eligible = (
    clinical_patient[clinical_patient["_PATIENT"].isin(expression_ids)]
    .set_index("_PATIENT")
)

train_ids, test_ids = train_test_split(
    eligible.index.to_numpy(),
    test_size=0.20,
    random_state=42,
)

assert set(train_ids).isdisjoint(test_ids)

print(f"Eligible patients: {len(eligible)}")
print(f"Train: {len(train_ids)} ({len(train_ids) / len(eligible):.1%})")
print(f"Test:  {len(test_ids)} ({len(test_ids) / len(eligible):.1%})")
print("Train/test patient overlap:", len(set(train_ids) & set(test_ids)))
print("Train OS event rate:", eligible.loc[train_ids, "OS"].mean())
print("Test OS event rate:", eligible.loc[test_ids, "OS"].mean())

Eligible patients: 10065
Train: 8052 (80.0%)
Test:  2013 (20.0%)
Train/test patient overlap: 0
Train OS event rate: 0.2976900149031297
Test OS event rate: 0.28912071535022354


In [7]:
patient_rows = adata_tcga.obs.groupby(
    adata_tcga.obs["patient_barcode"].astype(str),
    sort=False,
).indices

def mean_expression_by_patient(patient_ids):
    X_mean = np.empty((len(patient_ids), adata_tcga.n_vars), dtype=np.float32)

    for i, patient_id in enumerate(patient_ids):
        rows = patient_rows.get(str(patient_id))
        if rows is None:
            raise KeyError(f"No expression rows found for patient {patient_id}")

        X_mean[i] = np.asarray(
            adata_tcga.X[rows].mean(axis=0)
        ).reshape(-1)

    return X_mean

X_train = mean_expression_by_patient(train_ids)
X_test = mean_expression_by_patient(test_ids)

print("Train expression shape:", X_train.shape)
print("Test expression shape:", X_test.shape)
print("Train outcomes shape:", eligible.loc[train_ids, ["OS", "OS_time"]].shape)
print("Test outcomes shape:", eligible.loc[test_ids, ["OS", "OS_time"]].shape)

Train expression shape: (8052, 16335)
Test expression shape: (2013, 16335)
Train outcomes shape: (8052, 2)
Test outcomes shape: (2013, 2)


In [8]:
train_nan_by_gene = np.isnan(X_train).sum(axis=0)
test_nan_by_gene = np.isnan(X_test).sum(axis=0)

print("NaNs in train:", int(np.isnan(X_train).sum()))
print("NaNs in test:", int(np.isnan(X_test).sum()))
print("Genes with train NaNs:", int((train_nan_by_gene > 0).sum()))
print("Genes entirely missing in train:", int((train_nan_by_gene == len(train_ids)).sum()))
print("Infinite values in train:", int(np.isinf(X_train).sum()))
print("Infinite values in test:", int(np.isinf(X_test).sum()))

affected = np.flatnonzero(train_nan_by_gene > 0)[:10]
print("Example affected genes:", adata_tcga.var_names[affected].tolist())

NaNs in train: 191
NaNs in test: 38
Genes with train NaNs: 43
Genes entirely missing in train: 0
Infinite values in train: 0
Infinite values in test: 0
Example affected genes: ['AMY1A|276', 'APOBEC3A|200315', 'C14orf19|280655', 'CCL23|6368', 'CDR1|1038', 'CEACAM3|1084', 'CEACAM8|1088', 'CEP170L|645455', 'CTAGE4|100128553', 'CTAGE6|340307']


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

imputer = SimpleImputer(strategy="mean")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

pca = PCA(n_components=500, svd_solver="randomized", random_state=42)
PC_train = pca.fit_transform(X_train_scaled)
PC_test = pca.transform(X_test_scaled)

print("Training PCs:", PC_train.shape)
print("Test PCs:", PC_test.shape)
print("Training explained variance captured:", pca.explained_variance_ratio_.sum())
print("Remaining NaNs:", int(np.isnan(X_train_imputed).sum() + np.isnan(X_test_imputed).sum()))

Training PCs: (8052, 500)
Test PCs: (2013, 500)
Training explained variance captured: 0.8662027
Remaining NaNs: 0


In [10]:

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

pc_columns = [f"PC_{i}" for i in range(PC_train.shape[1])]

train_cox = pd.DataFrame(
    PC_train,
    columns=pc_columns,
    index=pd.Index(train_ids, name="patient_id"),
)
train_cox["OS_time"] = eligible.loc[train_ids, "OS_time"].to_numpy()
train_cox["OS"] = eligible.loc[train_ids, "OS"].to_numpy()

test_cox = pd.DataFrame(
    PC_test,
    columns=pc_columns,
    index=pd.Index(test_ids, name="patient_id"),
)
test_cox["OS_time"] = eligible.loc[test_ids, "OS_time"].to_numpy()
test_cox["OS"] = eligible.loc[test_ids, "OS"].to_numpy()

cox = CoxPHFitter()
cox.fit(train_cox, duration_col="OS_time", event_col="OS")

test_risk = cox.predict_partial_hazard(test_cox[pc_columns])
test_c_index = concordance_index(
    test_cox["OS_time"],
    -test_risk,
    test_cox["OS"],
)

print(f"Test C-index: {test_c_index:.4f}")
print(f"Penalizer: {cox.penalizer}")

Test C-index: 0.7683
Penalizer: 0.0


In [27]:
import joblib

model_bundle = {
    "imputer": imputer,
    "scaler": scaler,
    "pca": pca,
    "cox": cox,
    "gene_names": adata_tcga.var_names.to_list(),
    "train_patient_ids": list(train_ids),
    "test_patient_ids": list(test_ids),
    "test_c_index": float(test_c_index),
    "random_state": 42,
    "n_components": 500,
    "duration_col": "OS_time",
    "event_col": "OS",
}

joblib.dump(model_bundle, "tcga_cox_pca_model.joblib")
print("Saved tcga_cox_pca_model.joblib")

Saved tcga_cox_pca_model.joblib


In [28]:
saved = joblib.load("tcga_cox_pca_model.joblib")

print("Objects:", [key for key in ("imputer", "scaler", "pca", "cox") if key in saved])
print("PCA components:", saved["pca"].n_components_)
print("Test C-index:", saved["test_c_index"])

Objects: ['imputer', 'scaler', 'pca', 'cox']
PCA components: 500
Test C-index: 0.7682862130727549
